# CSCI 447/547 Hackathon 1.5:  K-Nearest Neighbors with Air Quality Data

This notebook is designed to be started during class and continued as a take-home activity

## How to use hackathon notebooks:

If the topic covered in a hackathon is new to you, work through the cells in order and read the explanation before running each code cell. You do not need to understand every detail of the hackathon on the first pass, instead, focus on the approach we take:

1. **Look at the data**
2. **Separate inputs from the target we want to predict**
3. **Split the data so we can test whether the model generalizes**
4. **Fit a model using the training data**
5. **Make predictions and evaluate them**
6. **Improve the model carefully <u>without</u> using the final evaluation data to make decisions**

We will work through the salary example together as a class. Pause before important code cells and think about what you expect to see. Please ask Lucy or a TA questions any time a term or line of code is unfamiliar.

After class, continue from wherever you left off. The existing explanations and code will be there to guide you through the process. Complete the marked answer sections, run every cell, and explain what the results mean in language that makes sense for you. Hackathons will not be graded, they are only to help you, and you will get out of them what you put into them.

<h4><span style="color:red">The goal of this notebook is NOT to memorize every function. It is to identify the processes we use in machine learning and be able to reuse them.</span></h4>

In this notebook, we will use **synthetic air-quality data** to understand how K-Nearest Neighbors (KNN) works.

We will ask two related questions:

1. **Classification:** Is today likely to be a **high-PM2.5 day**?
2. **Regression:** What PM2.5 concentration should we predict for today?

The goal is not to build the best air-quality model. Instead, the goal is to make the intuition behind KNN visible:

> **Find observations that look similar to the new observation, then use their outcomes to make a prediction.**

We will use weather variables that are easy to interpret:

- temperature
- relative humidity
- wind speed
- atmospheric pressure

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

np.random.seed(42)

ModuleNotFoundError: No module named 'pandas'

## 2. Create fake air-quality data

We will generate 500 synthetic days.

The PM2.5 values are designed to depend loosely on weather:

- warmer days tend to have somewhat higher PM2.5,
- drier days tend to have somewhat higher PM2.5,
- low-wind days tend to trap pollution,
- there is also random variation.

This is deliberately simplified. Real air-quality processes are much more complicated.

In [ ]:
n = 500

temperature = np.random.normal(70, 15, n).clip(25, 105)
humidity = np.random.normal(45, 18, n).clip(10, 95)
wind_speed = np.random.gamma(shape=2.2, scale=2.5, size=n).clip(0.5, 25)
pressure = np.random.normal(1013, 8, n)

# A synthetic PM2.5-generating relationship
pm25 = (
    8
    + 0.32 * (temperature - 60)
    - 0.10 * (humidity - 45)
    - 1.15 * (wind_speed - 5)
    + 0.08 * (pressure - 1013)
    + np.random.normal(0, 7, n)
)

# Add a few "smoke-event" days
smoke_event = np.random.binomial(1, 0.10, n)
pm25 += smoke_event * np.random.normal(35, 10, n)

pm25 = np.clip(pm25, 1, None)

df = pd.DataFrame({
    "temperature_F": temperature,
    "humidity_pct": humidity,
    "wind_speed_mph": wind_speed,
    "pressure_hPa": pressure,
    "pm25": pm25
})

# For classification, define a high-PM2.5 day.
# The cutoff is chosen for teaching purposes.
df["high_pm25"] = (df["pm25"] >= 35).astype(int)

df.head()

### Look at the data

In [ ]:
df.describe().round(2)

In [ ]:
print(df["high_pm25"].value_counts())
print()
print(df["high_pm25"].value_counts(normalize=True).round(3))

## 3. Visualize the basic idea

For now, use only two features:

- temperature
- wind speed

That lets us plot each day as a point in two-dimensional space.

Days near one another on this graph have similar temperature and wind conditions.

In [ ]:
plt.figure(figsize=(8, 6))

for label, name in [(0, "Lower PM2.5"), (1, "High PM2.5")]:
    subset = df[df["high_pm25"] == label]
    plt.scatter(
        subset["temperature_F"],
        subset["wind_speed_mph"],
        alpha=0.6,
        label=name
    )

plt.xlabel("Temperature (°F)")
plt.ylabel("Wind speed (mph)")
plt.title("Synthetic Air-Quality Days")
plt.legend()
plt.show()

NameError: name 'plt' is not defined

## 4. A new day arrives

Suppose today's weather is:

- **83°F**
- **28% humidity**
- **5 mph wind**
- **1012 hPa pressure**

We want to predict whether PM2.5 will be high.

KNN asks:

> Which previous days are most similar to today?

In [ ]:
today = pd.DataFrame({
    "temperature_F": [83],
    "humidity_pct": [28],
    "wind_speed_mph": [5],
    "pressure_hPa": [1012]
})

today

## 5. First KNN classifier

We'll start with all four features and use \(k=5\).

A KNN classifier:

1. calculates the distance from the new observation to every training observation,
2. finds the \(k\) closest observations,
3. lets those neighbors vote on the class.

In [ ]:
features = [
    "temperature_F",
    "humidity_pct",
    "wind_speed_mph",
    "pressure_hPa"
]

X = df[features]
y = df["high_pm25"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

prediction = knn.predict(today)[0]
probability = knn.predict_proba(today)[0, 1]

print("Predicted class:", "High PM2.5" if prediction == 1 else "Lower PM2.5")
print("Neighbor vote / estimated probability of high PM2.5:", round(probability, 2))

## 6. Inspect the actual nearest neighbors

One advantage of KNN is that we can literally inspect which observations produced the prediction.

In [ ]:
distances, indices = knn.kneighbors(today)

neighbors = X_train.iloc[indices[0]].copy()
neighbors["high_pm25"] = y_train.iloc[indices[0]].values
neighbors["pm25"] = df.loc[neighbors.index, "pm25"].values
neighbors["distance"] = distances[0]

neighbors.round(2)

### Discussion

Look at those five neighbors.

- Do they seem intuitively similar to today's weather?
- Are any variables dominating the notion of "similarity"?
- What does the majority vote predict?

## 7. The scaling problem

Euclidean distance is often used in KNN:

\[
d(x_i,x_j)=\sqrt{\sum_p(x_{ip}-x_{jp})^2}.
\]

But our variables have very different scales:

- pressure is around **1000**,
- temperature is around **20–100**,
- humidity is around **10–95**,
- wind speed is often around **1–15**.

That means a one-unit difference has very different implications across variables.

For distance-based algorithms, feature scaling is therefore extremely important.

In [ ]:
df[features].agg(["min", "max", "mean", "std"]).round(2)

## 8. Standardize the features

We can standardize each feature using

\[
z = \frac{x-\mu}{\sigma}.
\]

Now each feature is measured in standard-deviation units.

A `Pipeline` is the safest way to do this because the scaler is fit only on the training data.

In [ ]:
scaled_knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5)
)

scaled_knn.fit(X_train, y_train)

scaled_prediction = scaled_knn.predict(today)[0]
scaled_probability = scaled_knn.predict_proba(today)[0, 1]

print("Predicted class:", "High PM2.5" if scaled_prediction == 1 else "Lower PM2.5")
print("Estimated probability of high PM2.5:", round(scaled_probability, 2))

## 9. Compare performance with and without scaling

In [ ]:
unscaled_pred = knn.predict(X_test)
scaled_pred = scaled_knn.predict(X_test)

print("Unscaled KNN accuracy:", round(accuracy_score(y_test, unscaled_pred), 3))
print("Scaled KNN accuracy:  ", round(accuracy_score(y_test, scaled_pred), 3))

The exact difference will vary because the dataset is synthetic, but the important conceptual point is:

> **KNN predictions depend directly on distance, so the way features are scaled can change which points count as neighbors.**

## 10. Choosing \(k\)

The parameter \(k\) controls how local the prediction is.

- **Small \(k\)**: flexible and sensitive to individual observations
- **Large \(k\)**: smoother and more stable, but may ignore meaningful local structure

Let's evaluate several values of \(k\).

In [ ]:
k_values = range(1, 31)
accuracies = []

for k in k_values:
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k)
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    accuracies.append(accuracy_score(y_test, pred))

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), accuracies, marker="o")
plt.xlabel("k")
plt.ylabel("Test accuracy")
plt.title("Choosing the Number of Neighbors")
plt.show()

### A better practice: cross-validation

Choosing \(k\) based directly on the test set leaks information from the test set into model selection.

Instead, we can use cross-validation on the training data.

In [ ]:
cv_scores = []

for k in k_values:
    model = make_pipeline(
        StandardScaler(),
        KNeighborsClassifier(n_neighbors=k)
    )

    scores = cross_val_score(
        model,
        X_train,
        y_train,
        cv=5,
        scoring="accuracy"
    )

    cv_scores.append(scores.mean())

best_k = list(k_values)[np.argmax(cv_scores)]

print("Best k from 5-fold cross-validation:", best_k)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), cv_scores, marker="o")
plt.axvline(best_k, linestyle="--")
plt.xlabel("k")
plt.ylabel("Mean cross-validation accuracy")
plt.title("Selecting k with Cross-Validation")
plt.show()

## 11. Evaluate the selected classifier

In [ ]:
final_classifier = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=best_k)
)

final_classifier.fit(X_train, y_train)
final_pred = final_classifier.predict(X_test)

print("Test accuracy:", round(accuracy_score(y_test, final_pred), 3))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_pred,
    display_labels=["Lower PM2.5", "High PM2.5"]
)

plt.title("KNN Classification Confusion Matrix")
plt.show()

## 12. KNN regression

The same idea works when the outcome is continuous.

Instead of asking:

> Is PM2.5 high or low?

we can ask:

> What PM2.5 concentration should we predict?

A basic KNN regressor finds the \(k\) nearest observations and averages their outcomes:

\[
\hat{y}(x)=\frac{1}{k}\sum_{i \in N_k(x)}y_i.
\]

In [ ]:
y_reg = df["pm25"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X,
    y_reg,
    test_size=0.25,
    random_state=42
)

knn_reg = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=7)
)

knn_reg.fit(X_train_reg, y_train_reg)

today_pm25 = knn_reg.predict(today)[0]

print("Predicted PM2.5 for today:", round(today_pm25, 1), "µg/m³")

## 13. Evaluate the regression model

In [ ]:
reg_pred = knn_reg.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, reg_pred)
rmse = np.sqrt(mean_squared_error(y_test_reg, reg_pred))
r2 = r2_score(y_test_reg, reg_pred)

print("MAE: ", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R²:  ", round(r2, 3))

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test_reg, reg_pred, alpha=0.6)

low = min(y_test_reg.min(), reg_pred.min())
high = max(y_test_reg.max(), reg_pred.max())
plt.plot([low, high], [low, high], linestyle="--")

plt.xlabel("Observed PM2.5")
plt.ylabel("Predicted PM2.5")
plt.title("KNN Regression: Observed vs Predicted")
plt.show()

## 14. Optional extension: distance-weighted KNN

So far, every neighbor has had an equal vote.

But we might reasonably want **closer neighbors to matter more**.

Scikit-learn can do this with:

```python
weights="distance"
```

Try comparing uniform and distance-weighted KNN.

In [ ]:
uniform_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=7, weights="uniform")
)

distance_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=7, weights="distance")
)

uniform_model.fit(X_train_reg, y_train_reg)
distance_model.fit(X_train_reg, y_train_reg)

uniform_pred = uniform_model.predict(X_test_reg)
distance_pred = distance_model.predict(X_test_reg)

print(
    "Uniform-weight MAE:",
    round(mean_absolute_error(y_test_reg, uniform_pred), 2)
)

print(
    "Distance-weight MAE:",
    round(mean_absolute_error(y_test_reg, distance_pred), 2)
)

# Key Takeaways

KNN is conceptually simple:

1. Define what it means for two observations to be similar.
2. Find the \(k\) closest training observations.
3. Use their outcomes to make a prediction.

But several important machine-learning ideas appear immediately:

- **distance metrics matter**
- **feature scaling matters**
- \(k\) controls model complexity
- model selection should use **cross-validation**
- KNN can perform both **classification and regression**
- KNN makes very few assumptions about the functional relationship between predictors and outcomes

## Questions to think about

1. What happens when \(k=1\)?
2. What happens when \(k\) becomes very large?
3. Why does scaling matter more for KNN than for a decision tree?
4. Would Euclidean distance always be the best definition of similarity?
5. What happens as we add dozens or hundreds of features?
6. Would yesterday's air quality be a useful predictor? If so, how would that change the feature space?

# Student Challenge

Try making one or more of the following changes:

1. Change today's weather conditions and see how the prediction changes.
2. Compare \(k=1\), \(k=5\), and \(k=25\).
3. Remove one weather variable and re-evaluate the model.
4. Add yesterday's PM2.5 as a synthetic predictor.
5. Compare uniform voting with distance-weighted voting.
6. Change the threshold used to define "high PM2.5."
7. Explain why KNN may struggle when the number of predictors becomes very large.